# Information Content Metric (Data Enrichment)

This notebook demonstrates the process of "enriching" the knowledge base by adding calculated weights to symptom nodes. Each symptom in the graph is assigned a weight (Information Content - IC) calculated based on its frequency of occurrence across the entire disease corpus:

$$
IC = \log\left(\frac{Total\_Diseases}{Frequency + 1}\right)
$$

Why do we use this metric?

- **Balancing specificity**: Prevents generic symptoms (e.g., fever, fatigue) from overshadowing rarer, clinically more significant symptoms (e.g., a specific rash),

- **Penalizing non-specificity**: Automatically "penalizes" (assigns lower weight to) symptoms that appear in hundreds of different diseases, as they carry little diagnostic value,

- **Differentiation process**: Enables the system to logically distinguish specific clinical signs from general symptoms, which is the foundation for accurate diagnosis.

Example:
- Symptom "fever" (Frequency: 30) will have a low IC.
- Symptom "dysphonia" (Frequency: 2) will have a high IC.

## Package Installation

In [ ]:
%pip install neo4j python-dotenv

# Connecting to the Graph Database

In [ ]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()

neo4j_url = os.getenv("NEO4J_URL")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_username, neo4j_password))

## Total Number of Diseases (Total Corpus)

In [ ]:
total_diseases = 0

with driver.session() as session:
    result = session.run("""
     MATCH (d:Disease) 
     RETURN count(d) as total_diseases
    """)
    total_diseases = result.single()["total_diseases"]

In [ ]:
total_diseases

## Calculating and Writing the Weight (IDF) for Each Symptom

In [ ]:
with driver.session() as session:
    has_symptom_uri = "http://purl.obolibrary.org/obo/RO_0002452"

    result = session.run("""
        MATCH (s:Symptom)<-[r:n4sch__SCO_RESTRICTION]-(d:Disease)
        WHERE r.onPropertyURI = $has_symptom_uri
        
        WITH s, count(d) AS disease_frequency
        
        WITH s, log(toFloat($total_diseases) / (disease_frequency + 1)) AS ic_score
        
        SET s.weight = ic_score
        
        RETURN count(s) as symptoms_updated
    """, has_symptom_uri=has_symptom_uri, total_diseases=total_diseases)

    updated_count = result.single()["symptoms_updated"]
    print(f"Successfully calculated and written weights for {updated_count} symptoms.")

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (s:Symptom)
        WHERE s.weight IS NOT NULL
        RETURN s.n4sch__label[0] AS name, s.weight AS weight
        ORDER BY s.weight DESC
    """)

    for record in result:
        print(f"   - {record['name']}: {record['weight']:.4f}")